# Temporal Stability of 6D Head Pose Estimation in Video Sequences

**Authors:** Hugo García Souto and Adrián Martínez Balea

**Course:** Computer Vision II — Master's Degree in Artificial Intelligence, Universidade de Santiago de Compostela

In this project, we study video-based head pose estimation within the broader context of gaze and pose-based interaction. Our goal is not only to estimate the head orientation frame by frame, but also to analyze how stable these predictions are across time when the input is a video sequence.

We focus on 6D rotation-based head pose estimation, using 6DRepNet and its full-range extension as the main architectural reference. These methods are especially relevant for this task because they do not directly regress yaw, pitch and roll as three independent values. Instead, they predict a continuous 6D representation of a 3D rotation, which is later mapped into a valid rotation matrix. This makes the architecture well aligned with the actual geometry of the problem.

Our research question is:

**Can temporal post-processing improve the robustness and temporal stability of frame-by-frame 6D head pose estimation in video sequences?**

To answer this question, we will evaluate a frame-by-frame 6D head pose estimator on video sequences, compute angular and temporal metrics, and then compare the raw predictions with a temporally smoothed version of the same predictions. We expect smoothing to reduce prediction jitter, although it may also introduce lag during fast head rotations.


In [1]:
# Basic environment check

import os
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Current working directory:", os.getcwd())
print("Kaggle input directory exists:", Path("/kaggle/input").exists())

if TORCH_AVAILABLE:
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
else:
    print("PyTorch is not available.")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Current working directory: /kaggle/working
Kaggle input directory exists: True
PyTorch version: 2.10.0+cpu
CUDA available: False


## 1. Dataset and workspace inspection

Before implementing the head pose estimation pipeline, we first inspect the Kaggle workspace. This step helps us verify which datasets have been attached to the notebook and how their files are organized.

Since BIWI is a video-based dataset, we expect to work with sequences of frames and pose annotations. At this stage, we only inspect the available folders and file types. We do not need GPU acceleration yet, because no model inference is being performed.


In [7]:
# Inspect available Kaggle input datasets and file structure

from pathlib import Path
from collections import Counter

INPUT_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")

print("Input directory:", INPUT_DIR)
print("Working directory:", WORKING_DIR)
print()

if not INPUT_DIR.exists():
    print("The Kaggle input directory does not exist.")
else:
    dataset_dirs = sorted([p for p in INPUT_DIR.iterdir() if p.is_dir()])
    
    if not dataset_dirs:
        print("No datasets are currently attached to this notebook.")
    else:
        print(f"Found {len(dataset_dirs)} attached dataset(s):")
        for i, dataset_dir in enumerate(dataset_dirs, start=1):
            print(f"{i}. {dataset_dir.name}")
        
        print("\nTop-level contents:")
        for dataset_dir in dataset_dirs:
            print(f"\n--- {dataset_dir.name} ---")
            children = sorted(dataset_dir.iterdir())
            for child in children[:20]:
                kind = "DIR " if child.is_dir() else "FILE"
                print(f"[{kind}] {child.name}")
            if len(children) > 20:
                print(f"... {len(children) - 20} more item(s)")
        
        print("\nFile extension summary:")
        for dataset_dir in dataset_dirs:
            files = [p for p in dataset_dir.rglob("*") if p.is_file()]
            extensions = Counter(p.suffix.lower() if p.suffix else "[no extension]" for p in files)
            print(f"\n--- {dataset_dir.name} ---")
            print(f"Total files: {len(files)}")
            for ext, count in extensions.most_common(15):
                print(f"{ext}: {count}")

Input directory: /kaggle/input
Working directory: /kaggle/working

Found 2 attached dataset(s):
1. datasets
2. notebooks

Top-level contents:

--- datasets ---
[DIR ] androsstrk

--- notebooks ---
[DIR ] hortonhearsafoo
[DIR ] julythe2nd
[DIR ] kerneler

File extension summary:

--- datasets ---
Total files: 1718
.png: 1715
.txt: 3

--- notebooks ---
Total files: 0


## 2. Detailed inspection of the BIWI-like dataset

After attaching the BIWI dataset or a reduced BIWI subset, we inspect its internal structure more carefully. This is important because different public versions of the same dataset may organize frames, depth maps and pose annotations in slightly different ways.

At this stage, our goal is to identify three things:

1. where the RGB frames are stored,
2. where the head pose annotations are stored,
3. whether the files preserve a sequence structure that allows temporal analysis.


In [8]:
# Detailed dataset inspection

from pathlib import Path
from collections import Counter, defaultdict

INPUT_DIR = Path("/kaggle/input")

def describe_tree(root: Path, max_depth: int = 3, max_items_per_level: int = 30):
    root = Path(root)
    print(f"Root: {root}")
    print(f"Exists: {root.exists()}")
    print()
    
    if not root.exists():
        return
    
    def _walk(path: Path, depth: int):
        if depth > max_depth:
            return
        
        indent = "  " * depth
        children = sorted(path.iterdir())
        
        if depth == 0:
            print(f"{path.name}/")
        else:
            print(f"{indent}{path.name}/")
        
        shown = 0
        for child in children:
            if shown >= max_items_per_level:
                remaining = len(children) - shown
                print(f"{indent}  ... {remaining} more item(s)")
                break
            
            if child.is_dir():
                _walk(child, depth + 1)
            else:
                size_kb = child.stat().st_size / 1024
                print(f"{indent}  {child.name} ({size_kb:.1f} KB)")
            
            shown += 1
    
    _walk(root, 0)

dataset_dirs = sorted([p for p in INPUT_DIR.iterdir() if p.is_dir()])

print("Attached input folders:")
for p in dataset_dirs:
    print("-", p)
print()

for dataset_dir in dataset_dirs:
    print("=" * 80)
    describe_tree(dataset_dir, max_depth=3, max_items_per_level=20)
    print()

print("=" * 80)
print("Global file extension summary:")

for dataset_dir in dataset_dirs:
    files = [p for p in dataset_dir.rglob("*") if p.is_file()]
    extensions = Counter(p.suffix.lower() if p.suffix else "[no extension]" for p in files)
    
    print(f"\nDataset: {dataset_dir.name}")
    print(f"Total files: {len(files)}")
    for ext, count in extensions.most_common(20):
        print(f"  {ext}: {count}")

print("\nPotential annotation files:")
annotation_keywords = ["pose", "rot", "annot", "label", "ground", "gt", "txt", "csv", "mat"]
for dataset_dir in dataset_dirs:
    candidates = []
    for p in dataset_dir.rglob("*"):
        if p.is_file():
            name = p.name.lower()
            if any(key in name for key in annotation_keywords) or p.suffix.lower() in [".txt", ".csv", ".mat", ".json"]:
                candidates.append(p)
    
    print(f"\nDataset: {dataset_dir.name}")
    print(f"Found {len(candidates)} potential annotation file(s).")
    for p in candidates[:50]:
        print(" ", p.relative_to(dataset_dir))
    if len(candidates) > 50:
        print(f"  ... {len(candidates) - 50} more")

Attached input folders:
- /kaggle/input/datasets
- /kaggle/input/notebooks

Root: /kaggle/input/datasets
Exists: True

datasets/
  androsstrk/
    mini-biwi-dataset/
      Dataset/

Root: /kaggle/input/notebooks
Exists: True

notebooks/
  hortonhearsafoo/
  julythe2nd/
  kerneler/

Global file extension summary:

Dataset: datasets
Total files: 1718
  .png: 1715
  .txt: 3

Dataset: notebooks
Total files: 0

Potential annotation files:

Dataset: datasets
Found 3 potential annotation file(s).
  androsstrk/mini-biwi-dataset/Dataset/face_dataset_OF/11/angles.txt
  androsstrk/mini-biwi-dataset/Dataset/face_dataset_large/11/angles.txt
  androsstrk/mini-biwi-dataset/Dataset/face_dataset_ae/11/angles.txt

Dataset: notebooks
Found 0 potential annotation file(s).


## 3. Reading the mini-BIWI structure

The attached dataset is a reduced BIWI-like subset. It contains three variants of the same sequence, each one with RGB frames and an `angles.txt` file. Before choosing which version to use in the experiments, we inspect the number of frames, the image filenames, the image sizes and the format of the angle annotations.

This step is important because we should not assume the annotation order in advance. In head pose estimation, different datasets and implementations may store angles as yaw-pitch-roll, pitch-yaw-roll, or another convention. Therefore, we first inspect the raw files and then decide how to interpret them.


In [9]:
# Inspect the mini-BIWI dataset variants and annotation files

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

DATASET_ROOT = Path("/kaggle/input/datasets/androsstrk/mini-biwi-dataset/Dataset")

print("Dataset root:", DATASET_ROOT)
print("Exists:", DATASET_ROOT.exists())
print()

variant_dirs = sorted([p for p in DATASET_ROOT.iterdir() if p.is_dir()])
print("Dataset variants:")
for variant_dir in variant_dirs:
    print("-", variant_dir.name)

print("\nDetailed inspection:")

variant_summaries = []

for variant_dir in variant_dirs:
    print("=" * 80)
    print("Variant:", variant_dir.name)
    
    # The current mini dataset seems to contain subject/sequence folder "11"
    sequence_dirs = sorted([p for p in variant_dir.iterdir() if p.is_dir()])
    print("Sequence folders:", [p.name for p in sequence_dirs])
    
    for sequence_dir in sequence_dirs:
        image_paths = sorted(sequence_dir.glob("*.png"))
        angle_path = sequence_dir / "angles.txt"
        
        print(f"\nSequence: {sequence_dir.name}")
        print("Number of PNG frames:", len(image_paths))
        print("Angles file exists:", angle_path.exists())
        
        if image_paths:
            print("First image:", image_paths[0].name)
            print("Last image:", image_paths[-1].name)
            
            with Image.open(image_paths[0]) as img:
                print("First image size:", img.size, "| mode:", img.mode)
        
        if angle_path.exists():
            raw_lines = angle_path.read_text().splitlines()
            print("Number of annotation lines:", len(raw_lines))
            print("First 5 raw annotation lines:")
            for line in raw_lines[:5]:
                print("  ", line)
            
            try:
                angles = np.loadtxt(angle_path)
                if angles.ndim == 1:
                    angles = angles.reshape(1, -1)
                print("Loaded angles shape:", angles.shape)
                print("First 5 loaded rows:")
                print(angles[:5])
            except Exception as e:
                angles = None
                print("Could not load angles with numpy.loadtxt:", repr(e))
        
        variant_summaries.append({
            "variant": variant_dir.name,
            "sequence": sequence_dir.name,
            "num_images": len(image_paths),
            "angles_file": str(angle_path),
            "num_annotation_lines": len(raw_lines) if angle_path.exists() else None,
            "first_image": image_paths[0].name if image_paths else None,
            "last_image": image_paths[-1].name if image_paths else None,
        })

summary_df = pd.DataFrame(variant_summaries)
summary_df

Dataset root: /kaggle/input/datasets/androsstrk/mini-biwi-dataset/Dataset
Exists: True

Dataset variants:
- face_dataset_OF
- face_dataset_ae
- face_dataset_large

Detailed inspection:
Variant: face_dataset_OF
Sequence folders: ['11']

Sequence: 11
Number of PNG frames: 571
Angles file exists: True
First image: frame_00004_face_OF.png
Last image: frame_00574_face_OF.png
First image size: (100, 100) | mode: RGBA
Number of annotation lines: 571
First 5 raw annotation lines:
   00004	-2.9827	-11.2188	2.36576	378.343	196.763	91.2045	-67.5906	900.142
   00005	-2.90894	-11.2805	1.55915	378.696	196.304	91.7721	-68.3194	900.298
   00006	-2.69134	-11.3086	1.59942	378.404	196.204	91.3165	-68.477	900.313
   00007	-2.10872	-11.2541	2.34266	378.064	195.967	90.8335	-68.8835	900.786
   00008	-1.96196	-11.3352	2.03209	378.156	196.444	90.9254	-68.0984	900.271
Loaded angles shape: (571, 9)
First 5 loaded rows:
[[  4.       -2.9827  -11.2188    2.36576 378.343   196.763    91.2045
  -67.5906  900.142  ]


,variant,sequence,num_images,angles_file,num_annotation_lines,first_image,last_image
0,face_dataset_OF,11,571,/kaggle/input/datasets/androsstrk/mini-biwi-da...,571,frame_00004_face_OF.png,frame_00574_face_OF.png
1,face_dataset_ae,11,572,/kaggle/input/datasets/androsstrk/mini-biwi-da...,572,frame_00003_face_gray.png,frame_00574_face_gray.png
2,face_dataset_large,11,572,/kaggle/input/datasets/androsstrk/mini-biwi-da...,572,frame_00003_face_depth.png,frame_00574_face_depth.png
